# Phase 2 - Notebook 02: SplaTAM Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase2/02_splatam_architecture.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the overall SplaTAM system architecture
2. Learn how tracking works with Gaussian rendering
3. Understand the mapping module and Gaussian optimization
4. Know about silhouette-guided densification
5. See how tracking and mapping interact

**Estimated Time**: 75 minutes

**Prerequisites**: Notebook 01 (SLAM Basics), Phase 1 completion

---

## 1. SplaTAM Overview

**SplaTAM** = **Splat**, **Track** & **Map** 3D Gaussians

Key contributions:
1. First 3DGS-based dense RGB-D SLAM
2. Efficient differentiable rendering for tracking
3. Silhouette-guided Gaussian densification
4. Online map optimization

### System Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                         SplaTAM Pipeline                             │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│   RGB-D Frame (It, Dt)                                               │
│        │                                                             │
│        ▼                                                             │
│   ┌──────────────────────────────────────────────┐                   │
│   │              TRACKING MODULE                  │                   │
│   │  ┌────────────────────────────────────────┐  │                   │
│   │  │ 1. Render from pose estimate           │  │                   │
│   │  │ 2. Compare with observation            │  │                   │
│   │  │ 3. Optimize pose (gradient descent)    │  │                   │
│   │  └────────────────────────────────────────┘  │                   │
│   │            Output: Camera Pose Tt            │                   │
│   └──────────────────────┬───────────────────────┘                   │
│                          │                                            │
│                          ▼                                            │
│   ┌──────────────────────────────────────────────┐                   │
│   │            KEYFRAME SELECTION                 │                   │
│   │  • Check motion threshold                     │                   │
│   │  • Check overlap with existing keyframes     │                   │
│   └──────────────────────┬───────────────────────┘                   │
│                          │ (if new keyframe)                         │
│                          ▼                                            │
│   ┌──────────────────────────────────────────────┐                   │
│   │              MAPPING MODULE                   │                   │
│   │  ┌────────────────────────────────────────┐  │                   │
│   │  │ 1. Add new Gaussians from depth        │  │                   │
│   │  │ 2. Optimize Gaussians (render loss)    │  │                   │
│   │  │ 3. Densify using silhouette            │  │                   │
│   │  │ 4. Prune low-opacity Gaussians         │  │                   │
│   │  └────────────────────────────────────────┘  │                   │
│   │            Output: Updated Gaussian Map      │                   │
│   └──────────────────────────────────────────────┘                   │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Any

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("\nSplaTAM Architecture Tutorial")
print("=" * 40)

## 2. Configuration

SplaTAM uses configuration files to control system behavior. Let's define the key parameters.

In [ ]:
@dataclass
class TrackingConfig:
    """Configuration for camera tracking."""
    
    # Optimization
    iterations: int = 40
    learning_rate: float = 0.01
    
    # Loss weights
    use_depth: bool = True
    depth_weight: float = 0.5
    rgb_weight: float = 1.0
    
    # Convergence
    min_loss_change: float = 1e-6
    early_stop_patience: int = 5


@dataclass
class MappingConfig:
    """Configuration for Gaussian mapping."""
    
    # Optimization
    iterations: int = 60
    learning_rate_position: float = 0.001
    learning_rate_features: float = 0.01
    learning_rate_opacity: float = 0.05
    learning_rate_scaling: float = 0.005
    learning_rate_rotation: float = 0.001
    
    # Densification
    densify_interval: int = 20
    densify_grad_threshold: float = 0.0002
    min_opacity: float = 0.005
    
    # Gaussian initialization
    sh_degree: int = 0  # Start with SH degree 0 for speed
    initial_opacity: float = 0.5
    initial_scale: float = 0.01
    
    # Memory
    max_gaussians: int = 500000


@dataclass  
class KeyframeConfig:
    """Configuration for keyframe selection."""
    
    min_translation: float = 0.05  # meters
    min_rotation: float = 5.0  # degrees
    min_overlap: float = 0.7  # 70% overlap required
    max_overlap: float = 0.95  # Too similar if > 95%
    max_keyframes: int = 1000


# Create default configs
tracking_cfg = TrackingConfig()
mapping_cfg = MappingConfig()
keyframe_cfg = KeyframeConfig()

print("Tracking Config:")
print(f"  Iterations: {tracking_cfg.iterations}")
print(f"  Learning rate: {tracking_cfg.learning_rate}")
print(f"  Depth weight: {tracking_cfg.depth_weight}")

print("\nMapping Config:")
print(f"  Iterations: {mapping_cfg.iterations}")
print(f"  Max Gaussians: {mapping_cfg.max_gaussians}")
print(f"  Initial SH degree: {mapping_cfg.sh_degree}")

print("\nKeyframe Config:")
print(f"  Min translation: {keyframe_cfg.min_translation} m")
print(f"  Min rotation: {keyframe_cfg.min_rotation}°")

## 3. Gaussian Map Representation

The Gaussian map is the core data structure. Each Gaussian has:
- Position (xyz): 3D location
- Color (SH coefficients): View-dependent appearance
- Scale: 3D extent
- Rotation: Quaternion orientation
- Opacity: Transparency

In [ ]:
class GaussianMap:
    """
    3D Gaussian map for SLAM.
    
    Simplified version of the official 3DGS GaussianModel,
    adapted for online SLAM use.
    """
    
    def __init__(self, sh_degree: int = 0, device: str = "cuda"):
        self.sh_degree = sh_degree
        self.device = torch.device(device)
        
        # Gaussian parameters (initialized as empty)
        self._xyz = None           # [N, 3] positions
        self._features_dc = None   # [N, 1, 3] SH DC
        self._features_rest = None # [N, K, 3] SH rest
        self._scaling = None       # [N, 3] log-scale
        self._rotation = None      # [N, 4] quaternion
        self._opacity = None       # [N, 1] logit-opacity
        
        self._initialized = False
        
    @property
    def n_gaussians(self) -> int:
        """Number of Gaussians in the map."""
        if self._xyz is None:
            return 0
        return self._xyz.shape[0]
    
    @property
    def xyz(self) -> torch.Tensor:
        """Get positions."""
        return self._xyz
    
    @property
    def opacity(self) -> torch.Tensor:
        """Get opacity (sigmoid-activated)."""
        return torch.sigmoid(self._opacity)
    
    @property
    def scaling(self) -> torch.Tensor:
        """Get scales (exp-activated)."""
        return torch.exp(self._scaling)
    
    @property
    def rotation(self) -> torch.Tensor:
        """Get normalized rotation quaternions."""
        return F.normalize(self._rotation, dim=-1)
    
    def initialize_from_points(
        self,
        points: torch.Tensor,
        colors: torch.Tensor,
        scales: Optional[torch.Tensor] = None,
        initial_opacity: float = 0.5,
        initial_scale: float = 0.01,
    ):
        """
        Initialize Gaussian map from point cloud.
        
        Args:
            points: [N, 3] 3D positions
            colors: [N, 3] RGB colors in [0, 1]
            scales: [N, 3] optional initial scales
            initial_opacity: Default opacity value
            initial_scale: Default scale if not provided
        """
        N = points.shape[0]
        
        # Positions
        self._xyz = nn.Parameter(points.to(self.device))
        
        # Colors -> SH DC coefficients
        SH_C0 = 0.28209479177387814
        fused_color = (colors - 0.5) / SH_C0
        self._features_dc = nn.Parameter(
            fused_color.unsqueeze(1).to(self.device)
        )
        
        # Higher-order SH (zeros)
        n_sh_rest = (self.sh_degree + 1) ** 2 - 1
        self._features_rest = nn.Parameter(
            torch.zeros(N, max(n_sh_rest, 1), 3, device=self.device)
        )
        
        # Scales (stored as log)
        if scales is None:
            scales = torch.full((N, 3), initial_scale)
        self._scaling = nn.Parameter(
            torch.log(scales.to(self.device))
        )
        
        # Rotations (identity quaternions)
        rots = torch.zeros(N, 4, device=self.device)
        rots[:, 0] = 1.0  # w = 1, (x, y, z) = 0
        self._rotation = nn.Parameter(rots)
        
        # Opacities (stored as logit)
        opacity_logit = torch.log(
            torch.tensor(initial_opacity / (1 - initial_opacity))
        )
        self._opacity = nn.Parameter(
            torch.full((N, 1), opacity_logit, device=self.device)
        )
        
        self._initialized = True
        print(f"Initialized {N} Gaussians")
    
    def get_parameters(self) -> List[nn.Parameter]:
        """Get all learnable parameters."""
        return [
            self._xyz,
            self._features_dc,
            self._features_rest,
            self._scaling,
            self._rotation,
            self._opacity,
        ]
    
    def summary(self):
        """Print map summary."""
        print(f"\nGaussian Map Summary:")
        print(f"  Number of Gaussians: {self.n_gaussians:,}")
        print(f"  SH degree: {self.sh_degree}")
        if self._initialized:
            print(f"  Position range: [{self._xyz.min():.2f}, {self._xyz.max():.2f}]")
            print(f"  Mean opacity: {self.opacity.mean():.3f}")
            print(f"  Mean scale: {self.scaling.mean():.4f}")


# Create and test
gaussian_map = GaussianMap(sh_degree=0, device="cpu")

# Initialize with random points
test_points = torch.rand(1000, 3) * 2 - 1  # [-1, 1]^3
test_colors = torch.rand(1000, 3)

gaussian_map.initialize_from_points(test_points, test_colors)
gaussian_map.summary()

## 4. Tracking Module

The tracking module estimates camera pose by:
1. Rendering the Gaussian map from current pose estimate
2. Comparing rendered image with observed image
3. Optimizing pose to minimize photometric + depth loss

### Pose Parameterization

For gradient-based optimization, we use:
- **6D rotation** representation (first two columns of rotation matrix)
- **3D translation**

Total: 9 parameters to optimize

In [ ]:
class CameraTracker:
    """
    Camera tracking using Gaussian map rendering.
    
    Estimates camera pose by minimizing photometric difference
    between rendered and observed images.
    """
    
    def __init__(
        self,
        gaussian_map: GaussianMap,
        config: TrackingConfig,
    ):
        self.gaussian_map = gaussian_map
        self.config = config
        self.device = gaussian_map.device
        
        # Tracking history
        self.history = []
    
    def pose_to_params(self, pose: torch.Tensor) -> torch.Tensor:
        """
        Convert SE(3) pose to optimizable parameters.
        
        Uses 6D rotation representation + 3D translation.
        
        Args:
            pose: [4, 4] SE(3) matrix
        
        Returns:
            [9] parameter vector (6D rotation + 3D translation)
        """
        R = pose[:3, :3]
        t = pose[:3, 3]
        
        # 6D rotation: first two columns of R
        r6d = R[:, :2].reshape(-1)  # [6]
        
        return torch.cat([r6d, t])  # [9]
    
    def params_to_pose(self, params: torch.Tensor) -> torch.Tensor:
        """
        Convert parameters back to SE(3) pose.
        
        Uses Gram-Schmidt to ensure valid rotation matrix.
        
        Args:
            params: [9] parameter vector
        
        Returns:
            [4, 4] SE(3) matrix
        """
        r6d = params[:6].reshape(3, 2)
        t = params[6:9]
        
        # Gram-Schmidt orthogonalization
        a1 = r6d[:, 0]
        a2 = r6d[:, 1]
        
        b1 = F.normalize(a1, dim=0)
        b2 = a2 - (b1 @ a2) * b1
        b2 = F.normalize(b2, dim=0)
        b3 = torch.cross(b1, b2)
        
        R = torch.stack([b1, b2, b3], dim=1)
        
        # Build SE(3)
        pose = torch.eye(4, device=params.device)
        pose[:3, :3] = R
        pose[:3, 3] = t
        
        return pose
    
    def render(self, pose: torch.Tensor, H: int, W: int) -> Dict[str, torch.Tensor]:
        """
        Render Gaussians from given camera pose.
        
        Note: This is a simplified placeholder. The actual implementation
        uses the diff-gaussian-rasterization CUDA kernel.
        
        Args:
            pose: [4, 4] camera pose (world to camera)
            H, W: Image dimensions
        
        Returns:
            Dict with 'rgb' [3, H, W] and 'depth' [H, W]
        """
        # Placeholder - actual implementation uses CUDA rasterizer
        rgb = torch.zeros(3, H, W, device=self.device)
        depth = torch.ones(H, W, device=self.device)
        
        return {'rgb': rgb, 'depth': depth}
    
    def compute_loss(
        self,
        rendered: Dict[str, torch.Tensor],
        target_rgb: torch.Tensor,
        target_depth: Optional[torch.Tensor] = None,
        mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute tracking loss.
        
        Loss = rgb_weight * L1(rendered_rgb, target_rgb)
             + depth_weight * L1(rendered_depth, target_depth)
        """
        loss_dict = {}
        
        # RGB loss (L1)
        rgb_diff = torch.abs(rendered['rgb'] - target_rgb)
        if mask is not None:
            rgb_diff = rgb_diff * mask.unsqueeze(0)
        rgb_loss = rgb_diff.mean()
        loss_dict['rgb'] = rgb_loss.item()
        
        total_loss = self.config.rgb_weight * rgb_loss
        
        # Depth loss
        if self.config.use_depth and target_depth is not None:
            depth_diff = torch.abs(rendered['depth'] - target_depth)
            if mask is not None:
                depth_diff = depth_diff * mask
            depth_loss = depth_diff.mean()
            loss_dict['depth'] = depth_loss.item()
            total_loss = total_loss + self.config.depth_weight * depth_loss
        
        loss_dict['total'] = total_loss.item()
        return total_loss, loss_dict
    
    def track(
        self,
        rgb: torch.Tensor,
        depth: Optional[torch.Tensor],
        initial_pose: torch.Tensor,
        intrinsics: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, float, Dict[str, Any]]:
        """
        Estimate camera pose from observed image.
        
        Args:
            rgb: [3, H, W] or [H, W, 3] observed RGB
            depth: [H, W] observed depth (optional)
            initial_pose: [4, 4] initial pose estimate
            intrinsics: [3, 3] camera intrinsics
            mask: [H, W] valid pixel mask
        
        Returns:
            (optimized_pose, final_loss, debug_info)
        """
        # Ensure correct format
        if rgb.dim() == 3 and rgb.shape[-1] == 3:
            rgb = rgb.permute(2, 0, 1)
        
        H, W = rgb.shape[1], rgb.shape[2]
        
        # Convert pose to parameters
        pose_params = self.pose_to_params(initial_pose)
        pose_params = pose_params.clone().detach().requires_grad_(True)
        
        # Optimizer
        optimizer = torch.optim.Adam([pose_params], lr=self.config.learning_rate)
        
        # Optimization loop
        losses = []
        best_loss = float('inf')
        best_pose = initial_pose.clone()
        patience_counter = 0
        
        for i in range(self.config.iterations):
            optimizer.zero_grad()
            
            # Convert to pose
            current_pose = self.params_to_pose(pose_params)
            
            # Render
            rendered = self.render(current_pose, H, W)
            
            # Loss
            loss, loss_dict = self.compute_loss(rendered, rgb, depth, mask)
            
            # Backward
            loss.backward()
            optimizer.step()
            
            losses.append(loss.item())
            
            # Track best
            if loss.item() < best_loss:
                best_loss = loss.item()
                best_pose = current_pose.detach().clone()
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Early stopping
            if patience_counter >= self.config.early_stop_patience:
                break
            
            # Convergence check
            if len(losses) > 1:
                if abs(losses[-1] - losses[-2]) < self.config.min_loss_change:
                    break
        
        # Store history
        debug_info = {
            'losses': losses,
            'iterations': len(losses),
            'converged': patience_counter < self.config.early_stop_patience,
        }
        self.history.append(debug_info)
        
        return best_pose, best_loss, debug_info


# Test tracker
tracker = CameraTracker(gaussian_map, tracking_cfg)

# Test pose conversion
test_pose = torch.eye(4)
test_pose[:3, 3] = torch.tensor([1.0, 2.0, 3.0])  # Translation

params = tracker.pose_to_params(test_pose)
reconstructed_pose = tracker.params_to_pose(params)

print("Original pose:")
print(test_pose)
print("\nReconstructed pose:")
print(reconstructed_pose)
print(f"\nReconstruction error: {(test_pose - reconstructed_pose).abs().max():.6f}")

## 5. Mapping Module

The mapping module:
1. Adds new Gaussians from depth observations
2. Optimizes existing Gaussians using keyframes
3. Densifies under-reconstructed areas
4. Prunes low-opacity Gaussians

In [ ]:
class GaussianMapper:
    """
    Gaussian map management and optimization.
    
    Handles adding new Gaussians, optimization, densification, and pruning.
    """
    
    def __init__(self, config: MappingConfig, device: str = "cpu"):
        self.config = config
        self.device = torch.device(device)
        
        # Initialize Gaussian map
        self.gaussians = GaussianMap(sh_degree=config.sh_degree, device=device)
        self.optimizer = None
        
        # Statistics
        self.stats = {
            'total_added': 0,
            'total_pruned': 0,
            'optimization_steps': 0,
        }
    
    def unproject_depth(
        self,
        depth: torch.Tensor,
        intrinsics: torch.Tensor,
        pose: torch.Tensor,
        subsample: int = 4,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Unproject depth map to 3D points in world coordinates.
        
        Args:
            depth: [H, W] depth map
            intrinsics: [3, 3] camera intrinsics
            pose: [4, 4] camera pose (world to camera)
            subsample: Subsampling factor
        
        Returns:
            (points_world, valid_indices)
        """
        H, W = depth.shape
        device = depth.device
        
        # Create pixel grid
        v, u = torch.meshgrid(
            torch.arange(0, H, subsample, device=device),
            torch.arange(0, W, subsample, device=device),
            indexing='ij'
        )
        
        # Sample depth
        depth_sub = depth[::subsample, ::subsample]
        
        # Flatten
        v = v.reshape(-1).float()
        u = u.reshape(-1).float()
        z = depth_sub.reshape(-1)
        
        # Filter valid depth
        valid = z > 0
        v = v[valid]
        u = u[valid]
        z = z[valid]
        
        # Unproject to camera coordinates
        fx, fy = intrinsics[0, 0], intrinsics[1, 1]
        cx, cy = intrinsics[0, 2], intrinsics[1, 2]
        
        x_cam = (u - cx) * z / fx
        y_cam = (v - cy) * z / fy
        z_cam = z
        
        points_cam = torch.stack([x_cam, y_cam, z_cam], dim=1)  # [N, 3]
        
        # Transform to world coordinates
        # pose is world-to-camera, so world = pose^{-1} @ cam
        pose_inv = torch.linalg.inv(pose)
        R = pose_inv[:3, :3]
        t = pose_inv[:3, 3]
        
        points_world = (R @ points_cam.T).T + t  # [N, 3]
        
        return points_world, valid
    
    def add_gaussians_from_frame(
        self,
        rgb: torch.Tensor,
        depth: torch.Tensor,
        pose: torch.Tensor,
        intrinsics: torch.Tensor,
        subsample: int = 4,
    ) -> int:
        """
        Add new Gaussians from an RGB-D frame.
        
        Args:
            rgb: [3, H, W] or [H, W, 3] RGB image
            depth: [H, W] depth map
            pose: [4, 4] camera pose
            intrinsics: [3, 3] camera intrinsics
            subsample: Subsampling factor
        
        Returns:
            Number of Gaussians added
        """
        # Ensure correct format
        if rgb.dim() == 3 and rgb.shape[-1] == 3:
            rgb = rgb.permute(2, 0, 1)
        
        H, W = depth.shape
        
        # Unproject depth to 3D
        points, valid = self.unproject_depth(depth, intrinsics, pose, subsample)
        
        if len(points) == 0:
            return 0
        
        # Get colors for valid points
        rgb_sub = rgb[:, ::subsample, ::subsample]
        colors = rgb_sub.reshape(3, -1).T  # [N, 3]
        colors = colors[valid]
        
        # Check memory limit
        current_n = self.gaussians.n_gaussians
        max_add = self.config.max_gaussians - current_n
        if len(points) > max_add:
            # Random subsample
            idx = torch.randperm(len(points))[:max_add]
            points = points[idx]
            colors = colors[idx]
        
        if len(points) == 0:
            return 0
        
        # Initialize or add to map
        if not self.gaussians._initialized:
            self.gaussians.initialize_from_points(
                points, colors,
                initial_opacity=self.config.initial_opacity,
                initial_scale=self.config.initial_scale,
            )
            self._setup_optimizer()
        else:
            self._add_gaussians(points, colors)
        
        self.stats['total_added'] += len(points)
        return len(points)
    
    def _add_gaussians(
        self,
        points: torch.Tensor,
        colors: torch.Tensor,
    ):
        """Add new Gaussians to existing map."""
        N = points.shape[0]
        
        # Create new parameters
        SH_C0 = 0.28209479177387814
        new_xyz = points.to(self.device)
        new_features_dc = ((colors - 0.5) / SH_C0).unsqueeze(1).to(self.device)
        new_features_rest = torch.zeros(
            N, self.gaussians._features_rest.shape[1], 3, device=self.device
        )
        new_scaling = torch.full((N, 3), np.log(self.config.initial_scale), device=self.device)
        new_rotation = torch.zeros(N, 4, device=self.device)
        new_rotation[:, 0] = 1.0
        opacity_logit = np.log(self.config.initial_opacity / (1 - self.config.initial_opacity))
        new_opacity = torch.full((N, 1), opacity_logit, device=self.device)
        
        # Concatenate
        self.gaussians._xyz = nn.Parameter(
            torch.cat([self.gaussians._xyz.data, new_xyz], dim=0)
        )
        self.gaussians._features_dc = nn.Parameter(
            torch.cat([self.gaussians._features_dc.data, new_features_dc], dim=0)
        )
        self.gaussians._features_rest = nn.Parameter(
            torch.cat([self.gaussians._features_rest.data, new_features_rest], dim=0)
        )
        self.gaussians._scaling = nn.Parameter(
            torch.cat([self.gaussians._scaling.data, new_scaling], dim=0)
        )
        self.gaussians._rotation = nn.Parameter(
            torch.cat([self.gaussians._rotation.data, new_rotation], dim=0)
        )
        self.gaussians._opacity = nn.Parameter(
            torch.cat([self.gaussians._opacity.data, new_opacity], dim=0)
        )
        
        # Recreate optimizer
        self._setup_optimizer()
    
    def _setup_optimizer(self):
        """Setup optimizer with per-parameter learning rates."""
        cfg = self.config
        param_groups = [
            {'params': [self.gaussians._xyz], 'lr': cfg.learning_rate_position, 'name': 'xyz'},
            {'params': [self.gaussians._features_dc], 'lr': cfg.learning_rate_features, 'name': 'f_dc'},
            {'params': [self.gaussians._features_rest], 'lr': cfg.learning_rate_features / 20, 'name': 'f_rest'},
            {'params': [self.gaussians._scaling], 'lr': cfg.learning_rate_scaling, 'name': 'scaling'},
            {'params': [self.gaussians._rotation], 'lr': cfg.learning_rate_rotation, 'name': 'rotation'},
            {'params': [self.gaussians._opacity], 'lr': cfg.learning_rate_opacity, 'name': 'opacity'},
        ]
        self.optimizer = torch.optim.Adam(param_groups)
    
    def optimize(self, keyframes: List[Any], n_iterations: Optional[int] = None) -> float:
        """
        Optimize Gaussians using keyframes.
        
        Note: Simplified placeholder. Actual implementation renders from
        keyframe poses and computes photometric + depth loss.
        """
        if not self.gaussians._initialized or len(keyframes) == 0:
            return float('inf')
        
        n_iters = n_iterations or self.config.iterations
        total_loss = 0.0
        
        for i in range(n_iters):
            self.optimizer.zero_grad()
            
            # In real implementation:
            # 1. Random keyframe selection
            # 2. Render from keyframe pose
            # 3. Compute loss vs keyframe image
            # 4. Backward and step
            
            # Placeholder loss
            loss = torch.tensor(0.0, requires_grad=True)
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            self.stats['optimization_steps'] += 1
            
            # Densification
            if i > 0 and i % self.config.densify_interval == 0:
                self._prune_low_opacity()
        
        return total_loss / n_iters if n_iters > 0 else 0.0
    
    def _prune_low_opacity(self):
        """Prune Gaussians with opacity below threshold."""
        with torch.no_grad():
            opacity = self.gaussians.opacity.squeeze()
            mask = opacity > self.config.min_opacity
            
            n_pruned = self.gaussians.n_gaussians - mask.sum().item()
            if n_pruned > 0:
                self.gaussians._xyz = nn.Parameter(self.gaussians._xyz.data[mask])
                self.gaussians._features_dc = nn.Parameter(self.gaussians._features_dc.data[mask])
                self.gaussians._features_rest = nn.Parameter(self.gaussians._features_rest.data[mask])
                self.gaussians._scaling = nn.Parameter(self.gaussians._scaling.data[mask])
                self.gaussians._rotation = nn.Parameter(self.gaussians._rotation.data[mask])
                self.gaussians._opacity = nn.Parameter(self.gaussians._opacity.data[mask])
                
                self._setup_optimizer()
                self.stats['total_pruned'] += n_pruned
    
    def get_stats(self) -> Dict[str, Any]:
        """Get mapping statistics."""
        return {
            **self.stats,
            'n_gaussians': self.gaussians.n_gaussians,
        }


# Test mapper
mapper = GaussianMapper(mapping_cfg, device="cpu")

# Create synthetic frame
H, W = 480, 640
test_rgb = torch.rand(3, H, W)
test_depth = torch.rand(H, W) * 5 + 0.5  # 0.5 to 5.5 meters
test_pose = torch.eye(4)
test_intrinsics = torch.tensor([
    [500.0, 0.0, 320.0],
    [0.0, 500.0, 240.0],
    [0.0, 0.0, 1.0]
])

n_added = mapper.add_gaussians_from_frame(
    test_rgb, test_depth, test_pose, test_intrinsics, subsample=8
)

print(f"\nAdded {n_added} Gaussians from frame")
print(f"Statistics: {mapper.get_stats()}")
mapper.gaussians.summary()

## 6. Silhouette-Guided Densification

SplaTAM introduces **silhouette-guided densification** to handle under-reconstructed regions:

1. Render the silhouette (where Gaussians cover the image)
2. Compare with observed depth silhouette
3. Add new Gaussians in uncovered regions

This is more principled than the gradient-based densification in original 3DGS.

In [ ]:
def compute_silhouette_mask(
    rendered_depth: torch.Tensor,
    observed_depth: torch.Tensor,
    depth_threshold: float = 0.1,
) -> torch.Tensor:
    """
    Compute mask of under-reconstructed regions.
    
    Args:
        rendered_depth: [H, W] rendered depth from Gaussians
        observed_depth: [H, W] observed depth
        depth_threshold: Threshold for considering a pixel covered
    
    Returns:
        [H, W] binary mask (True = needs more Gaussians)
    """
    # Valid observed depth
    valid_obs = observed_depth > 0
    
    # Rendered silhouette (where we have Gaussians)
    rendered_silhouette = rendered_depth < float('inf')
    
    # Under-reconstructed: valid observation but no rendered coverage
    under_reconstructed = valid_obs & ~rendered_silhouette
    
    return under_reconstructed


# Visualize silhouette-guided densification concept
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Create example scenario
H, W = 100, 150

# Observed depth (full scene)
observed = np.zeros((H, W))
observed[20:80, 20:130] = 1.0  # Object at 1m
observed[30:50, 30:50] = 0.5   # Closer object at 0.5m
observed[40:70, 80:120] = 1.5  # Further object at 1.5m

# Rendered depth (partial coverage - Gaussians only cover some areas)
rendered = np.full((H, W), np.inf)
rendered[25:75, 25:70] = 0.95  # Only covers left part
rendered[30:50, 30:50] = 0.5   # Covers closer object

# Compute masks
valid_obs = observed > 0
has_gaussians = rendered < np.inf
need_densification = valid_obs & ~has_gaussians

# Plot
axes[0, 0].imshow(observed, cmap='viridis')
axes[0, 0].set_title('Observed Depth')
axes[0, 0].axis('off')

axes[0, 1].imshow(np.where(rendered < np.inf, rendered, 0), cmap='viridis')
axes[0, 1].set_title('Rendered Depth (Current Gaussians)')
axes[0, 1].axis('off')

axes[0, 2].imshow(valid_obs, cmap='gray')
axes[0, 2].set_title('Observed Silhouette')
axes[0, 2].axis('off')

axes[1, 0].imshow(has_gaussians, cmap='gray')
axes[1, 0].set_title('Rendered Silhouette')
axes[1, 0].axis('off')

axes[1, 1].imshow(need_densification, cmap='Reds')
axes[1, 1].set_title('Under-reconstructed Regions\n(Need Densification)')
axes[1, 1].axis('off')

# Overlay
overlay = np.zeros((H, W, 3))
overlay[..., 1] = has_gaussians.astype(float) * 0.5  # Green: covered
overlay[..., 0] = need_densification.astype(float)    # Red: needs more
axes[1, 2].imshow(overlay)
axes[1, 2].set_title('Overlay\nGreen=Covered, Red=Need More')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

# Statistics
total_pixels = H * W
valid_pixels = valid_obs.sum()
covered_pixels = (valid_obs & has_gaussians).sum()
uncovered_pixels = need_densification.sum()

print(f"\nCoverage Statistics:")
print(f"  Total pixels: {total_pixels}")
print(f"  Valid observed: {valid_pixels} ({100*valid_pixels/total_pixels:.1f}%)")
print(f"  Covered by Gaussians: {covered_pixels} ({100*covered_pixels/valid_pixels:.1f}% of valid)")
print(f"  Need densification: {uncovered_pixels} ({100*uncovered_pixels/valid_pixels:.1f}% of valid)")

## 7. Complete SLAM Loop

Now let's see how tracking and mapping work together in the main SLAM loop.

In [ ]:
@dataclass
class Keyframe:
    """A keyframe with associated data."""
    frame_id: int
    pose: torch.Tensor
    rgb: torch.Tensor
    depth: torch.Tensor
    intrinsics: torch.Tensor


class SplaTAMSystem:
    """
    Complete SplaTAM SLAM system.
    
    Integrates tracking, mapping, and keyframe management.
    """
    
    def __init__(
        self,
        tracking_config: TrackingConfig = None,
        mapping_config: MappingConfig = None,
        keyframe_config: KeyframeConfig = None,
        device: str = "cpu",
    ):
        self.tracking_cfg = tracking_config or TrackingConfig()
        self.mapping_cfg = mapping_config or MappingConfig()
        self.keyframe_cfg = keyframe_config or KeyframeConfig()
        self.device = torch.device(device)
        
        # Initialize mapper (which contains the Gaussian map)
        self.mapper = GaussianMapper(self.mapping_cfg, device)
        
        # Tracker will be created after first frame
        self.tracker = None
        
        # Keyframe management
        self.keyframes: List[Keyframe] = []
        self.last_keyframe_pose = None
        
        # Trajectory
        self.poses: List[torch.Tensor] = []
        
        # Statistics
        self.frame_count = 0
        self.tracking_losses: List[float] = []
    
    def process_frame(
        self,
        rgb: torch.Tensor,
        depth: torch.Tensor,
        intrinsics: torch.Tensor,
        gt_pose: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Dict[str, Any]]:
        """
        Process a single RGB-D frame.
        
        Args:
            rgb: [3, H, W] or [H, W, 3] RGB image
            depth: [H, W] depth map
            intrinsics: [3, 3] camera intrinsics
            gt_pose: Optional ground truth pose for initialization
        
        Returns:
            (estimated_pose, info_dict)
        """
        self.frame_count += 1
        
        # First frame: initialize
        if self.frame_count == 1:
            return self._initialize_from_first_frame(rgb, depth, intrinsics, gt_pose)
        
        # TRACKING: Estimate current pose
        initial_pose = self.poses[-1].clone()  # Previous pose as initial estimate
        
        estimated_pose, tracking_loss, tracking_info = self.tracker.track(
            rgb, depth, initial_pose, intrinsics
        )
        
        self.poses.append(estimated_pose)
        self.tracking_losses.append(tracking_loss)
        
        # KEYFRAME SELECTION
        is_keyframe = self._should_add_keyframe(estimated_pose)
        
        info = {
            'tracking_loss': tracking_loss,
            'tracking_iterations': tracking_info['iterations'],
            'is_keyframe': is_keyframe,
        }
        
        if is_keyframe:
            # MAPPING: Add new Gaussians and optimize
            n_added = self.mapper.add_gaussians_from_frame(
                rgb, depth, estimated_pose, intrinsics
            )
            
            # Add keyframe
            kf = Keyframe(
                frame_id=self.frame_count,
                pose=estimated_pose.clone(),
                rgb=rgb.clone(),
                depth=depth.clone(),
                intrinsics=intrinsics.clone(),
            )
            self.keyframes.append(kf)
            self.last_keyframe_pose = estimated_pose.clone()
            
            # Optimize map
            mapping_loss = self.mapper.optimize(
                self.keyframes[-5:],  # Use last 5 keyframes
                n_iterations=self.mapping_cfg.iterations,
            )
            
            info['n_gaussians_added'] = n_added
            info['mapping_loss'] = mapping_loss
            info['n_keyframes'] = len(self.keyframes)
        
        info['n_gaussians'] = self.mapper.gaussians.n_gaussians
        
        return estimated_pose, info
    
    def _initialize_from_first_frame(
        self,
        rgb: torch.Tensor,
        depth: torch.Tensor,
        intrinsics: torch.Tensor,
        gt_pose: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Dict[str, Any]]:
        """Initialize system from first frame."""
        
        # Use GT pose or identity
        pose = gt_pose if gt_pose is not None else torch.eye(4, device=self.device)
        
        # Add Gaussians from first frame
        n_added = self.mapper.add_gaussians_from_frame(
            rgb, depth, pose, intrinsics
        )
        
        # Create tracker
        self.tracker = CameraTracker(self.mapper.gaussians, self.tracking_cfg)
        
        # Add first keyframe
        kf = Keyframe(
            frame_id=self.frame_count,
            pose=pose.clone(),
            rgb=rgb.clone(),
            depth=depth.clone(),
            intrinsics=intrinsics.clone(),
        )
        self.keyframes.append(kf)
        self.last_keyframe_pose = pose.clone()
        self.poses.append(pose)
        
        info = {
            'initialized': True,
            'n_gaussians_added': n_added,
            'n_gaussians': self.mapper.gaussians.n_gaussians,
            'is_keyframe': True,
            'n_keyframes': 1,
        }
        
        print(f"Initialized SplaTAM with {n_added} Gaussians")
        
        return pose, info
    
    def _should_add_keyframe(self, current_pose: torch.Tensor) -> bool:
        """Decide if current frame should be a keyframe."""
        if self.last_keyframe_pose is None:
            return True
        
        # Compute relative motion
        relative = current_pose @ torch.linalg.inv(self.last_keyframe_pose)
        
        # Translation
        translation = relative[:3, 3].norm().item()
        
        # Rotation (from trace)
        R = relative[:3, :3]
        trace = R.trace()
        cos_angle = torch.clamp((trace - 1) / 2, -1, 1)
        rotation_deg = torch.acos(cos_angle).item() * 180 / np.pi
        
        # Check thresholds
        return (
            translation > self.keyframe_cfg.min_translation or
            rotation_deg > self.keyframe_cfg.min_rotation
        )
    
    def get_trajectory(self) -> torch.Tensor:
        """Get estimated camera trajectory."""
        if not self.poses:
            return torch.empty(0, 4, 4)
        return torch.stack(self.poses)
    
    def summary(self):
        """Print system summary."""
        print(f"\nSplaTAM System Summary:")
        print(f"  Frames processed: {self.frame_count}")
        print(f"  Keyframes: {len(self.keyframes)}")
        print(f"  Gaussians: {self.mapper.gaussians.n_gaussians:,}")
        if self.tracking_losses:
            print(f"  Mean tracking loss: {np.mean(self.tracking_losses):.4f}")
        print(f"  Mapper stats: {self.mapper.get_stats()}")

In [ ]:
# Simulate running SplaTAM on a sequence

# Create system
splatam = SplaTAMSystem(
    tracking_config=TrackingConfig(iterations=10),
    mapping_config=MappingConfig(iterations=10, max_gaussians=10000),
    keyframe_config=KeyframeConfig(min_translation=0.1, min_rotation=5.0),
    device="cpu",
)

# Simulate a sequence
n_frames = 20
H, W = 240, 320

# Camera intrinsics
K = torch.tensor([
    [250.0, 0.0, 160.0],
    [0.0, 250.0, 120.0],
    [0.0, 0.0, 1.0]
])

# Simulate circular trajectory
print("Processing frames...")
for i in range(n_frames):
    # Create synthetic frame
    rgb = torch.rand(3, H, W)
    depth = torch.rand(H, W) * 3 + 1  # 1 to 4 meters
    
    # Ground truth pose (circular motion)
    angle = 2 * np.pi * i / n_frames
    gt_pose = torch.eye(4)
    gt_pose[0, 3] = np.cos(angle) * 2
    gt_pose[2, 3] = np.sin(angle) * 2
    
    # Process frame
    pose, info = splatam.process_frame(rgb, depth, K, gt_pose)
    
    if i % 5 == 0 or info.get('is_keyframe', False):
        status = "[KF]" if info.get('is_keyframe', False) else "    "
        n_gauss = info.get('n_gaussians', 0)
        print(f"  Frame {i:3d} {status} | Gaussians: {n_gauss:,}")

splatam.summary()

In [ ]:
# Visualize the trajectory

trajectory = splatam.get_trajectory()
positions = trajectory[:, :3, 3].numpy()

keyframe_positions = [kf.pose[:3, 3].numpy() for kf in splatam.keyframes]
keyframe_positions = np.array(keyframe_positions)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trajectory plot
ax1 = axes[0]
ax1.plot(positions[:, 0], positions[:, 2], 'b-', linewidth=2, label='Trajectory')
ax1.scatter(positions[:, 0], positions[:, 2], c=np.arange(len(positions)), cmap='viridis', s=30)
ax1.scatter(keyframe_positions[:, 0], keyframe_positions[:, 2], c='red', s=100, marker='^', label='Keyframes', zorder=5)
ax1.scatter([0], [0], c='green', s=200, marker='*', label='Origin', zorder=6)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Z (m)')
ax1.set_title('Camera Trajectory (Top View)')
ax1.legend()
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# Tracking loss plot
ax2 = axes[1]
ax2.plot(splatam.tracking_losses, 'b-', linewidth=1)
ax2.set_xlabel('Frame')
ax2.set_ylabel('Tracking Loss')
ax2.set_title('Tracking Loss Over Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Summary

### SplaTAM Architecture

| Component | Function | Key Features |
|-----------|----------|-------------|
| **Gaussian Map** | Scene representation | Optimizable positions, colors, scales |
| **Tracking** | Pose estimation | Render-and-compare, 6D+3D parameterization |
| **Mapping** | Map optimization | Per-parameter LR, densification, pruning |
| **Keyframes** | Memory management | Motion-based selection, covisibility |

### Key Insights

1. **Differentiable rendering** enables gradient-based pose optimization
2. **Silhouette-guided densification** adds Gaussians where needed
3. **Keyframe-based optimization** balances quality and speed
4. **Online operation** requires careful memory management

---

## What's Next?

**[03_map_initialization.ipynb](./03_map_initialization.ipynb)** - Deep dive into Gaussian map initialization from depth

---

## References

1. SplaTAM: https://spla-tam.github.io/
2. SplaTAM Paper: https://arxiv.org/abs/2312.02126
3. SplaTAM Code: https://github.com/spla-tam/SplaTAM
4. 3D Gaussian Splatting: https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/